# DATA PREPARATION AND BASELINE FORECASTING

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("ggplot")

In [2]:
df = pd.read_excel("../data/final_anomaly_dataset (1).xlsx")

In [3]:
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,IQR_Anomaly,IsolationForest,ZScore_Anomaly
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,0,0,False
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,0,0,False
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,0,0,False
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,0,0,False
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,0,0,False


In [4]:
df.tail()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,IQR_Anomaly,IsolationForest,ZScore_Anomaly
73095,2024-01-01,S005,P0016,Furniture,East,96,8,127,18.46,73.73,20,Snowy,0,72.45,Winter,0,0,False
73096,2024-01-01,S005,P0017,Toys,North,313,51,101,48.43,82.57,10,Cloudy,0,83.78,Autumn,0,0,False
73097,2024-01-01,S005,P0018,Clothing,West,278,36,151,39.65,11.11,10,Rainy,0,10.91,Winter,0,0,False
73098,2024-01-01,S005,P0019,Toys,East,374,264,21,270.52,53.14,20,Rainy,0,55.80,Spring,0,0,False
73099,2024-01-01,S005,P0020,Groceries,East,117,6,165,2.33,78.39,20,Rainy,1,79.52,Spring,0,0,False


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                73100 non-null  object 
 1   Store ID            73100 non-null  object 
 2   Product ID          73100 non-null  object 
 3   Category            73100 non-null  object 
 4   Region              73100 non-null  object 
 5   Inventory Level     73100 non-null  int64  
 6   Units Sold          73100 non-null  int64  
 7   Units Ordered       73100 non-null  int64  
 8   Demand Forecast     73100 non-null  float64
 9   Price               73100 non-null  float64
 10  Discount            73100 non-null  int64  
 11  Weather Condition   73100 non-null  object 
 12  Holiday/Promotion   73100 non-null  int64  
 13  Competitor Pricing  73100 non-null  float64
 14  Seasonality         73100 non-null  object 
 15  IQR_Anomaly         73100 non-null  int64  
 16  Isol

In [6]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 73100
Columns: 18


In [7]:
df["Date"].head()

0    2022-01-01
1    2022-01-01
2    2022-01-01
3    2022-01-01
4    2022-01-01
Name: Date, dtype: object

In [8]:
df["Date"] = pd.to_datetime(df["Date"])

In [9]:
df["Date"].dtype

dtype('<M8[ns]')

In [10]:
df = df.sort_values("Date")

In [11]:
print(df["Date"].head())

print(df["Date"].tail())

0    2022-01-01
72   2022-01-01
71   2022-01-01
70   2022-01-01
69   2022-01-01
Name: Date, dtype: datetime64[ns]
73027   2024-01-01
73026   2024-01-01
73025   2024-01-01
73035   2024-01-01
73099   2024-01-01
Name: Date, dtype: datetime64[ns]


In [12]:
print(
    "Chronological Order:",
    df["Date"].is_monotonic_increasing
)

Chronological Order: True


In [13]:
print("First Date:", df["Date"].iloc[0])
print("Last Date:", df["Date"].iloc[-1])
print("Chronological Order:", df["Date"].is_monotonic_increasing)

First Date: 2022-01-01 00:00:00
Last Date: 2024-01-01 00:00:00
Chronological Order: True


# DATA CLEANING AND PREPROCESSING

In [14]:
print("Missing Values:", df.isnull().sum().sum())

Missing Values: 0


In [15]:
print("Duplicate Records:", df.duplicated().sum())

Duplicate Records: 0


In [16]:
print(df.dtypes)

Date                  datetime64[ns]
Store ID                      object
Product ID                    object
Category                      object
Region                        object
Inventory Level                int64
Units Sold                     int64
Units Ordered                  int64
Demand Forecast              float64
Price                        float64
Discount                       int64
Weather Condition             object
Holiday/Promotion              int64
Competitor Pricing           float64
Seasonality                   object
IQR_Anomaly                    int64
IsolationForest                int64
ZScore_Anomaly                  bool
dtype: object


In [17]:
print(df[["Date","Units Sold"]].head())

         Date  Units Sold
0  2022-01-01         127
72 2022-01-01          56
71 2022-01-01           9
70 2022-01-01          46
69 2022-01-01         104


# SIMPLE MOVING AVERAGE BASELINE FORECASTING

In [19]:
daily_sales = (
    df.groupby("Date")["Units Sold"]
    .sum()
    .reset_index()
)

In [20]:
daily_sales.head()

,Date,Units Sold
0,2022-01-01,14484
1,2022-01-02,13415
2,2022-01-03,13681
3,2022-01-04,14084
4,2022-01-05,12572


In [21]:
daily_sales.shape

(731, 2)

In [22]:
print("Date Data Type:",daily_sales["Date"].dtype)

Date Data Type: datetime64[ns]


In [23]:
daily_sales.set_index("Date", inplace=True)

In [24]:
daily_sales.head()

,Units Sold
Date,
2022-01-01,14484
2022-01-02,13415
2022-01-03,13681
2022-01-04,14084
2022-01-05,12572


In [25]:
daily_sales.describe()

,Units Sold
count,731.000000
mean,13646.487004
std,1031.958966
min,10642.000000
25%,12981.000000
50%,13673.000000
75%,14323.500000
max,17239.000000


In [26]:
daily_sales["SMA_7"] = (
    daily_sales["Units Sold"]
    .rolling(window=7)
    .mean()
)

In [27]:
daily_sales.head(10)

,Units Sold,SMA_7
Date,,
2022-01-01,14484,NaN
2022-01-02,13415,NaN
2022-01-03,13681,NaN
2022-01-04,14084,NaN
2022-01-05,12572,NaN
2022-01-06,12563,NaN
2022-01-07,11826,13232.142857
2022-01-08,13958,13157.000000
2022-01-09,15896,13511.428571


In [28]:
print("SMA Values Generated:",daily_sales["SMA_7"].notna().sum())

SMA Values Generated: 725


In [29]:
print("Missing SMA Values:",daily_sales["SMA_7"].isnull().sum())

Missing SMA Values: 6


In [30]:
daily_sales.tail(10)

,Units Sold,SMA_7
Date,,
2023-12-23,13335,13492.571429
2023-12-24,13433,13321.142857
2023-12-25,12723,13234.285714
2023-12-26,15858,13820.285714
2023-12-27,13023,13542.857143
2023-12-28,16271,14111.285714
2023-12-29,13368,14001.571429
2023-12-30,13156,13976.000000
2023-12-31,11208,13658.142857
